# 05 · Hipótesis exploratorias

Estas tres hipótesis se plantearon **después** de ver los resultados de H1 a H5, por eso son exploratorias. Se registraron en `docs/bitacora.md` con su parámetro exacto **antes** de calcularlas, y se informan todas con el resultado que salga.

| ID | Idea |
| --- | --- |
| **HC2 · Para qué** | Los estudiantes de mayor ESCS usan la IA más para investigar y aprender, y los de menor ESCS más para redactar borradores |
| **HC3 · A quién llega** | La IA llega más a quienes rinden más |
| **HC4 · La misma escuela** | La brecha de uso según ESCS se mantiene dentro de una misma escuela |

Como son tres hipótesis, se usa la corrección de Bonferroni: se considera resultado claro si p < 0,05 / 3 = 0,0167. Todo vale solo para quienes contestaron las cuatro preguntas de IA y son asociaciones, no causas.

In [1]:
from pathlib import Path
from math import erfc, sqrt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raiz = Path.cwd()
while not (raiz / "README.md").exists():
    if raiz == raiz.parent:
        raise FileNotFoundError("No se encontró README.md: ejecutá el notebook dentro del proyecto.")
    raiz = raiz.parent

archivo = raiz / "data" / "processed" / "argentina_pisa2025_analitico.csv"
carpeta_figuras = raiz / "reports" / "figures"
carpeta_tablas = raiz / "reports" / "tables"

datos = pd.read_csv(archivo)
muestra_ia = datos[datos["ESCS"].notna() & (datos["ia_estado"] == "respondio_las_4")].copy()
print("Muestra (ESCS y las 4 preguntas de IA):", len(muestra_ia))

Muestra (ESCS y las 4 preguntas de IA): 7191


## 1. Funciones de estimación

Las mismas del notebook 03 (peso final, 80 pesos replicados y 10 valores plausibles).

In [2]:
N_REPLICAS = 80
FACTOR_FAY = 0.5
DIVISOR = N_REPLICAS * (1 - FACTOR_FAY) ** 2     # 80 * 0.25 = 20

pesos_replicados = []
for i in range(1, 81):
    pesos_replicados.append("W_FSTURWT" + str(i))

pv_ciencias = []
pv_matematica = []
pv_lectura = []
for i in range(1, 11):
    pv_ciencias.append("PV" + str(i) + "SCIE")
    pv_matematica.append("PV" + str(i) + "MATH")
    pv_lectura.append("PV" + str(i) + "READ")

print("Divisor de la fórmula:", DIVISOR)

Divisor de la fórmula: 20.0


In [3]:
def media_pv(datos, pv, peso):
    media = np.average(datos[pv].values, weights=datos[peso].values)
    return np.array([media])


def media_variable(datos, pv, peso, variable):
    # pv no se usa acá: se deja para que todas las funciones reciban los mismos argumentos
    media = np.average(datos[variable].values, weights=datos[peso].values)
    return np.array([media])


def diferencia_pv(datos, pv, peso, columna, grupo_a, grupo_b):
    a = datos[datos[columna] == grupo_a]
    b = datos[datos[columna] == grupo_b]
    media_a = np.average(a[pv].values, weights=a[peso].values)
    media_b = np.average(b[pv].values, weights=b[peso].values)
    return np.array([media_a - media_b])


def diferencia_variable(datos, pv, peso, columna, grupo_a, grupo_b, variable):
    # pv no se usa acá, igual que en media_variable
    a = datos[datos[columna] == grupo_a]
    b = datos[datos[columna] == grupo_b]
    media_a = np.average(a[variable].values, weights=a[peso].values)
    media_b = np.average(b[variable].values, weights=b[peso].values)
    return np.array([media_a - media_b])


def regresion_pv(datos, pv, peso, predictores):
    y = datos[pv].values
    w = datos[peso].values
    columnas = [np.ones(len(datos))]
    for predictor in predictores:
        columnas.append(datos[predictor].values)
    X = np.column_stack(columnas)
    raiz_w = np.sqrt(w)
    coeficientes = np.linalg.lstsq(X * raiz_w[:, None], y * raiz_w, rcond=None)[0]
    return coeficientes

In [4]:
def estimar(datos, funcion, pvs, **argumentos):
    estimaciones = []
    varianzas = []

    for pv in pvs:
        estimacion = funcion(datos, pv, "W_FSTUWT", **argumentos)

        replicas = []
        for peso_replicado in pesos_replicados:
            replicas.append(funcion(datos, pv, peso_replicado, **argumentos))
        replicas = np.array(replicas)

        varianza_muestral = np.sum((replicas - estimacion) ** 2, axis=0) / DIVISOR
        estimaciones.append(estimacion)
        varianzas.append(varianza_muestral)

    estimaciones = np.array(estimaciones)
    varianzas = np.array(varianzas)
    m = len(pvs)

    estimacion_final = estimaciones.mean(axis=0)
    varianza_muestral = varianzas.mean(axis=0)
    if m > 1:
        varianza_entre_pv = estimaciones.var(axis=0, ddof=1)
    else:
        varianza_entre_pv = 0

    varianza_total = varianza_muestral + (1 + 1 / m) * varianza_entre_pv
    return estimacion_final, np.sqrt(varianza_total)

In [5]:
resultados = []

def guardar(hipotesis, parametro, n, estimacion, error, nota=""):
    z = estimacion / error
    p = erfc(abs(z) / sqrt(2))
    if p < 0.0001:
        texto_p = "<0,0001"
    else:
        texto_p = str(round(p, 4))
    inferior = estimacion - 1.96 * error
    superior = estimacion + 1.96 * error
    if p < 0.05 and (inferior > 0 or superior < 0):
        veredicto = "compatible con una asociación"
    else:
        veredicto = "evidencia insuficiente"
    resultados.append({
        "hipotesis": hipotesis, "parametro": parametro, "n": n,
        "estimacion": round(estimacion, 3), "error_estandar": round(error, 3),
        "ic95_inferior": round(inferior, 3), "ic95_superior": round(superior, 3),
        "p": p, "p_texto": texto_p, "veredicto": veredicto, "nota": nota,
    })

## 2. HC2 · ¿Para qué se usa la IA según el nivel socioeconómico?

**Índice de tipo de uso** de cada estudiante: promedio de la frecuencia de *investigar un tema* y *ayudar a aprender*, menos la frecuencia de *redactar borradores*. Un valor positivo significa que usa más la IA para investigar y aprender que para redactar.

**Parámetro:** diferencia del índice entre el cuartil 4 y el cuartil 1 de ESCS.

In [6]:
investigar_y_aprender = (muestra_ia["ST438Q02DA"] + muestra_ia["ST438Q04DA"]) / 2
muestra_ia["indice_tipo_uso"] = investigar_y_aprender - muestra_ia["ST438Q03DA"]

print("Índice de tipo de uso: mínimo", muestra_ia["indice_tipo_uso"].min(), "| máximo", muestra_ia["indice_tipo_uso"].max())

# Promedio del índice por cuartil (con pesos y réplicas)
promedios = []
for cuartil in [1.0, 2.0, 3.0, 4.0]:
    grupo = muestra_ia[muestra_ia["cuartil_escs"] == cuartil]
    media, error = estimar(grupo, media_variable, [None], variable="indice_tipo_uso")
    promedios.append((cuartil, round(media[0], 3), round(error[0], 3)))
print("Cuartil, promedio, error estándar:", promedios)

estimacion, error = estimar(muestra_ia, diferencia_variable, [None],
                            columna="cuartil_escs", grupo_a=4.0, grupo_b=1.0, variable="indice_tipo_uso")
guardar("HC2", "Índice de tipo de uso (investigar y aprender menos redactar): cuartil 4 menos cuartil 1 de ESCS",
        len(muestra_ia), estimacion[0], error[0], nota="exploratoria")
print(pd.DataFrame(resultados)[["parametro", "n", "estimacion", "error_estandar", "ic95_inferior", "ic95_superior", "p_texto"]].to_string())

Índice de tipo de uso: mínimo -4.0 | máximo 4.0
Cuartil, promedio, error estándar: [(1.0, np.float64(0.482), np.float64(0.043)), (2.0, np.float64(0.487), np.float64(0.046)), (3.0, np.float64(0.491), np.float64(0.041)), (4.0, np.float64(0.563), np.float64(0.038))]


                                                                                         parametro     n  estimacion  error_estandar  ic95_inferior  ic95_superior p_texto
0  Índice de tipo de uso (investigar y aprender menos redactar): cuartil 4 menos cuartil 1 de ESCS  7191       0.082           0.057          -0.03          0.194  0.1526


**Lectura de HC2.** El índice promedio es 0,48 en el cuartil 1, 0,49 en el 2, 0,49 en el 3 y 0,56 en el 4. La diferencia entre el cuartil 4 y el 1 es +0,08 (IC 95 %: -0,03 a 0,19; p = 0,15). Va en la dirección esperada, pero es chica y no se distingue de cero. **Veredicto: evidencia insuficiente.** Con estos datos **no se puede sostener** que los estudiantes de distinto nivel socioeconómico usen la IA para cosas distintas: la brecha de uso es más de cuánto que de para qué.

## 3. HC3 · ¿La IA llega más a quienes rinden más?

Se arman los cuartiles de rendimiento en ciencias **con cada valor plausible y con cada peso** (así los cortes también entran en el error). Se compara la proporción que usa IA *semanal o diariamente en alguna actividad* entre el cuartil 4 y el cuartil 1 de rendimiento.

Atención: no se sabe qué causa qué. Que quienes rinden más usen más IA no significa que la IA los haga rendir más.

In [7]:
ia_4 = ["ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA"]
muestra_ia["usa_semanal_alguna"] = (muestra_ia[ia_4].max(axis=1) >= 4).astype(int)

def corte_ponderado(valores, pesos, proporcion):
    orden = np.argsort(valores.values)
    valores_ordenados = valores.values[orden]
    pesos_ordenados = pesos.values[orden]
    acumulado = np.cumsum(pesos_ordenados) / pesos_ordenados.sum()
    posicion = np.searchsorted(acumulado, proporcion)
    return valores_ordenados[posicion]


def brecha_por_rendimiento(datos, pv, peso, variable):
    corte_bajo = corte_ponderado(datos[pv], datos[peso], 0.25)
    corte_alto = corte_ponderado(datos[pv], datos[peso], 0.75)
    cuartil_bajo = datos[datos[pv] <= corte_bajo]
    cuartil_alto = datos[datos[pv] > corte_alto]
    media_bajo = np.average(cuartil_bajo[variable].values, weights=cuartil_bajo[peso].values)
    media_alto = np.average(cuartil_alto[variable].values, weights=cuartil_alto[peso].values)
    return np.array([media_alto - media_bajo])


estimacion, error = estimar(muestra_ia, brecha_por_rendimiento, pv_ciencias, variable="usa_semanal_alguna")
guardar("HC3", "Proporción que usa IA semanal o diaria en alguna actividad: cuartil 4 de ciencias menos cuartil 1 (0,10 = 10 puntos porcentuales)",
        len(muestra_ia), estimacion[0], error[0], nota="exploratoria")

estimacion, error = estimar(muestra_ia, brecha_por_rendimiento, pv_ciencias, variable="ia_promedio")
guardar("HC3", "Frecuencia promedio de IA (1 a 5): cuartil 4 de ciencias menos cuartil 1",
        len(muestra_ia), estimacion[0], error[0], nota="exploratoria")

tabla_hc3 = pd.DataFrame(resultados)
tabla_hc3 = tabla_hc3[tabla_hc3["hipotesis"] == "HC3"]
print(tabla_hc3[["parametro", "n", "estimacion", "error_estandar", "ic95_inferior", "ic95_superior", "p_texto"]].to_string())

                                                                                                                           parametro     n  estimacion  error_estandar  ic95_inferior  ic95_superior  p_texto
1  Proporción que usa IA semanal o diaria en alguna actividad: cuartil 4 de ciencias menos cuartil 1 (0,10 = 10 puntos porcentuales)  7191       0.166           0.027          0.113          0.220  <0,0001
2                                                           Frecuencia promedio de IA (1 a 5): cuartil 4 de ciencias menos cuartil 1  7191       0.379           0.076          0.230          0.529  <0,0001


**Lectura de HC3.** El cuartil de mayor rendimiento en ciencias usa IA semanal o diariamente en alguna actividad **16,6 puntos porcentuales más** que el de menor rendimiento (IC 95 %: 11,3 a 22,0), y su frecuencia promedio es 0,38 puntos más alta en la escala de 1 a 5. Es clara incluso con la corrección de Bonferroni. **Veredicto: evidencia compatible con una asociación.** Atención: esto describe a quienes contestaron las preguntas de IA, y los que no contestaron son los que menos rinden (HC1), así que el panorama del conjunto puede ser distinto. Tampoco se sabe qué causa qué: puede que quienes rinden más se animen más a usar IA, o que otras características de los estudiantes expliquen las dos cosas.

## 4. HC4 · ¿La brecha se mantiene dentro de una misma escuela?

Se compara el coeficiente de ESCS sobre la frecuencia promedio de IA:
- **total:** entre todos los estudiantes;
- **dentro de escuela:** a cada estudiante se le resta el promedio (ponderado) de su escuela, tanto en ESCS como en IA, de modo que solo se comparan estudiantes de la misma escuela.

In [8]:
def pendiente_total(datos, pv, peso, variable, predictor):
    w = datos[peso]
    x = datos[predictor]
    y = datos[variable]
    media_x = np.average(x, weights=w)
    media_y = np.average(y, weights=w)
    coeficiente = np.sum(w * (x - media_x) * (y - media_y)) / np.sum(w * (x - media_x) ** 2)
    return np.array([coeficiente])


def pendiente_dentro_de_escuela(datos, pv, peso, variable, predictor):
    w = datos[peso]
    x = datos[predictor]
    y = datos[variable]
    escuela = datos["CNTSCHID"]

    peso_escuela = w.groupby(escuela).transform("sum")
    media_x_escuela = (w * x).groupby(escuela).transform("sum") / peso_escuela
    media_y_escuela = (w * y).groupby(escuela).transform("sum") / peso_escuela

    x_dentro = x - media_x_escuela
    y_dentro = y - media_y_escuela
    coeficiente = np.sum(w * x_dentro * y_dentro) / np.sum(w * x_dentro ** 2)
    return np.array([coeficiente])


total, error_total = estimar(muestra_ia, pendiente_total, [None], variable="ia_promedio", predictor="ESCS")
dentro, error_dentro = estimar(muestra_ia, pendiente_dentro_de_escuela, [None], variable="ia_promedio", predictor="ESCS")

guardar("HC4", "ESCS sobre frecuencia de IA, total (puntos de la escala 1 a 5 por unidad de ESCS)",
        len(muestra_ia), total[0], error_total[0], nota="exploratoria")
guardar("HC4", "ESCS sobre frecuencia de IA, dentro de escuela",
        len(muestra_ia), dentro[0], error_dentro[0], nota="exploratoria")

tabla_hc4 = pd.DataFrame(resultados)
tabla_hc4 = tabla_hc4[tabla_hc4["hipotesis"] == "HC4"]
print(tabla_hc4[["parametro", "n", "estimacion", "error_estandar", "ic95_inferior", "ic95_superior", "p_texto"]].to_string())
print("Escuelas en la muestra:", muestra_ia["CNTSCHID"].nunique())
print("Estudiantes por escuela (mediana):", muestra_ia["CNTSCHID"].value_counts().median())

                                                                           parametro     n  estimacion  error_estandar  ic95_inferior  ic95_superior  p_texto
3  ESCS sobre frecuencia de IA, total (puntos de la escala 1 a 5 por unidad de ESCS)  7191       0.185           0.024          0.139          0.231  <0,0001
4                                     ESCS sobre frecuencia de IA, dentro de escuela  7191       0.080           0.028          0.026          0.134   0.0036
Escuelas en la muestra: 404
Estudiantes por escuela (mediana): 18.0


**Lectura de HC4.** El coeficiente de ESCS sobre la frecuencia de IA es 0,185 puntos de la escala por unidad de ESCS en el total (IC 95 %: 0,139 a 0,231) y 0,080 dentro de una misma escuela (0,026 a 0,134; p = 0,0036). **Veredicto: evidencia compatible con una asociación**, también dentro de las escuelas, pero a menos de la mitad (43 %) del tamaño. Es decir, **más de la mitad de la brecha de uso entre niveles socioeconómicos se da entre escuelas** (a qué escuela se asiste) y menos de la mitad dentro de ellas.

## 5. Tabla de resultados

In [9]:
tabla_final = pd.DataFrame(resultados)
tabla_final["clara_con_bonferroni"] = tabla_final["p"] < 0.05 / 3
tabla_final.to_csv(carpeta_tablas / "05_hipotesis_exploratorias.csv", index=False)
print(tabla_final[["hipotesis", "parametro", "estimacion", "ic95_inferior", "ic95_superior", "p_texto", "clara_con_bonferroni"]].to_string())

  hipotesis                                                                                                                          parametro  estimacion  ic95_inferior  ic95_superior  p_texto  clara_con_bonferroni
0       HC2                                    Índice de tipo de uso (investigar y aprender menos redactar): cuartil 4 menos cuartil 1 de ESCS       0.082         -0.030          0.194   0.1526                 False
1       HC3  Proporción que usa IA semanal o diaria en alguna actividad: cuartil 4 de ciencias menos cuartil 1 (0,10 = 10 puntos porcentuales)       0.166          0.113          0.220  <0,0001                  True
2       HC3                                                           Frecuencia promedio de IA (1 a 5): cuartil 4 de ciencias menos cuartil 1       0.379          0.230          0.529  <0,0001                  True
3       HC4                                                  ESCS sobre frecuencia de IA, total (puntos de la escala 1 a 5 por unidad de

## 6. Resumen

| Hipótesis | Resultado |
| --- | --- |
| HC2 · Para qué | Evidencia insuficiente: no se sostiene que el tipo de uso cambie con el nivel socioeconómico |
| HC3 · A quién llega | Compatible con una asociación: más uso entre quienes rinden más (+16,6 puntos porcentuales) |
| HC4 · La misma escuela | Compatible con una asociación dentro de escuela, a menos de la mitad del tamaño total |

Son hipótesis exploratorias, planteadas después de ver H1 a H5, con corrección de Bonferroni para tres. Son asociaciones y valen para quienes contestaron las preguntas de IA.